# 🎹 Song → Piano MIDI (free Google GPU)

Turns any song into a MIDI file your player piano can play - in about a minute instead of 15.

**First time only:** menu **Runtime → Change runtime type → T4 GPU → Save**.

**Then:** press **Runtime → Run all** (or Ctrl+F9). Step 3 shows an **upload button** - pick your mp3(s).
When each song is done, its `.mid` downloads automatically. Drag it into **Piano Jukebox → My Music**.

Your song is only processed inside your own Colab session and is deleted when the session ends.

In [ ]:
#@title 1. Set up (takes ~1 minute)
import subprocess, sys, os, torch
gpu = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if gpu else "NONE - turn it on: Runtime > Change runtime type > T4 GPU, then Run all again")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "piano_transcription_inference", "demucs", "audioread", "mido", "scipy"], check=True)
ckpt_dir = os.path.expanduser("~/piano_transcription_inference_data")
ckpt = os.path.join(ckpt_dir, "note_F1=0.9677_pedal_F1=0.9186.pth")
if not os.path.exists(ckpt) or os.path.getsize(ckpt) < 1.6e8:
    os.makedirs(ckpt_dir, exist_ok=True)
    print("Downloading the piano model (165 MB)...")
    subprocess.run(["wget", "-q", "-O", ckpt,
                    "https://zenodo.org/record/4034264/files/CRNN_note_F1%3D0.9677_pedal_F1%3D0.9186.pth?download=1"], check=True)
print("Ready.")

In [ ]:
%%writefile song2midi.py
"""Song -> piano MIDI.

  python song2midi.py                    record from the mic, press Enter to stop
  python song2midi.py song.mp3           transcribe an audio file (cleaner than the mic)
  python song2midi.py --mode pop ...     Pop/Film mode: strip drums etc. first (see below)
  python song2midi.py --seconds 60       record for a fixed time instead of waiting for Enter

Modes
  piano  (default) best for songs that are already mostly piano
  pop    for pop / film songs: splits the song into vocals, drums, bass and
         "other" (keys, guitar, strings) with Demucs, throws the drums away,
         transcribes the rest and cleans up stray notes
         --parts vocals,other,bass   choose which parts become piano notes
                                     (e.g. --parts vocals for just the tune)

Output goes to the "output" folder next to this script. Drop the .mid into
Piano Jukebox -> My Music.
"""
import argparse
import contextlib
import io
import os
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import numpy as np

SR = 16000  # the transcription model works at 16 kHz mono
HERE = Path(__file__).resolve().parent
OUT = HERE / "output"
PARTS = ("vocals", "other", "bass", "drums")


def quiet():
    return contextlib.redirect_stdout(io.StringIO())


def record(seconds=None):
    """Returns (audio[channels, samples], samplerate)."""
    import sounddevice as sd

    dev = sd.query_devices(kind="input")
    rate = int(dev["default_samplerate"])
    print(f"Mic: {dev['name']}")
    chunks, level = [], [0.0]

    def cb(data, frames, t, status):
        chunks.append(data[:, 0].copy())
        level[0] = float(np.abs(data).max())

    input("Start the song on your phone, hold it near the mic, then press Enter to start recording...")
    print("Recording" + (f" for {seconds}s" if seconds else " - press Enter to stop") + "\n")
    start = time.time()
    with sd.InputStream(samplerate=rate, channels=1, dtype="float32", callback=cb):
        if os.name == "nt":
            import msvcrt
            while True:
                el = time.time() - start
                bar = "#" * int(min(level[0], 1.0) * 40)
                sys.stdout.write(f"\r  {el:6.1f}s  [{bar:<40}]")
                sys.stdout.flush()
                if seconds and el >= seconds:
                    break
                if msvcrt.kbhit() and msvcrt.getwch() in "\r\n":
                    break
                time.sleep(0.1)
        elif seconds:
            time.sleep(seconds)
        else:
            input()
    print("\n")
    if not chunks:
        sys.exit("Nothing was recorded - check the mic.")
    audio = np.concatenate(chunks)
    if np.abs(audio).max() < 0.01:
        print("Warning: the recording is very quiet. Turn the phone up or move it closer.")
    return audio[None, :], rate


def load_file(path):
    import librosa
    audio, rate = librosa.load(path, sr=None, mono=False)
    if audio.ndim == 1:
        audio = audio[None, :]
    return audio, rate


_models = {}
import threading
_model_lock = threading.RLock()  # the server may ask for a model from two threads at once


def demucs_model():
    with _model_lock:
        return _load_demucs()


def _load_demucs():
    if "demucs" not in _models:
        from demucs.pretrained import get_model
        with quiet():
            m = get_model("htdemucs")
        m.eval()
        _models["demucs"] = m
    return _models["demucs"]


def piano_model():
    with _model_lock:
        return _load_piano()


def _load_piano():
    if "piano" not in _models:
        import torch
        from piano_transcription_inference import PianoTranscription
        with quiet():
            _models["piano"] = PianoTranscription(
                device=torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    return _models["piano"]


MODE_NAMES = {"piano": "Piano", "pop": "Pop/Film", "auto": "Auto"}
PART_NAMES = {"vocals": "Melody (vocals)", "other": "Chords & instruments", "bass": "Bass", "drums": "Drums"}


def analyse_stems(stems, sources, sr):
    """How much of the song each part is playing, and its share of the total energy."""
    win = sr  # 1-second windows
    frames = stems.shape[-1] // win
    if frames < 1:
        return {s: {"active": 1.0, "share": 0.25} for s in sources}
    mono = stems.mean(1)[:, : frames * win].reshape(len(sources), frames, win)  # [src, frame, sample]
    rms = np.sqrt((mono ** 2).mean(-1))                                          # [src, frame]
    loud = np.percentile(rms.sum(0), 95) + 1e-9   # reference: the loud parts of the whole song
    energy = (rms ** 2).sum(1)
    return {s: {"active": float((rms[i] > 0.08 * loud).mean()),
                "share": float(energy[i] / (energy.sum() + 1e-12))} for i, s in enumerate(sources)}


def auto_parts(info):
    """Pick what to turn into piano. Returns (parts or None for 'use the original audio', reason)."""
    v, o, b, d = (info[k] for k in ("vocals", "other", "bass", "drums"))
    # Mostly a keyboard/instrumental track with no real drums or singing: the untouched audio
    # transcribes best (separation only adds artifacts).
    if d["share"] < 0.03 and v["active"] < 0.10 and o["share"] > 0.7:
        return None, "mostly piano/keys with no drums or singing - using the original audio"
    parts, why = [], []
    if v["active"] >= 0.15:
        parts.append("vocals"); why.append(f"singing in {v['active']:.0%} of the song -> melody")
    if o["active"] >= 0.15:
        parts.append("other"); why.append(f"instruments in {o['active']:.0%} -> chords")
    if "other" not in parts or o["share"] < 0.10:
        if b["active"] >= 0.15:
            parts.append("bass"); why.append("little else carrying the harmony -> keeping bass")
    else:
        why.append("bass left out (it muddies the chords)")
    if not parts:
        parts, why = ["vocals", "other"], ["nothing stood out - using melody + chords"]
    why.append("drums removed")
    return parts, "; ".join(why)


@contextlib.contextmanager
def transcribe_progress(report):
    """Temporarily make the piano model report how many audio segments it has done."""
    import torch
    from piano_transcription_inference import inference
    from piano_transcription_inference.pytorch_utils import move_data_to_device
    original = inference.forward

    def forward(model, x, batch_size):
        out, device = {}, next(model.parameters()).device
        for i in range(0, len(x), batch_size):
            with torch.no_grad():
                model.eval()
                res = model(move_data_to_device(x[i:i + batch_size], device))
            for k, v in res.items():
                out.setdefault(k, []).append(v.data.cpu().numpy())
            report(min(1.0, (i + batch_size) / len(x)))
        return {k: np.concatenate(v, axis=0) for k, v in out.items()}

    inference.forward = forward
    try:
        yield
    finally:
        inference.forward = original


def separate(audio, rate, parts, status=print, progress=True, on_progress=None):
    """Split with Demucs and return the chosen parts mixed to mono at `rate`.
    parts may be "auto", in which case they're chosen from what the song contains.
    Returns (mono, rate, description)."""
    import librosa
    import torch
    from demucs.apply import apply_model

    status("Splitting the song into parts (Demucs)...")
    model = demucs_model()
    sr = model.samplerate
    wav = librosa.resample(audio, orig_sr=rate, target_sr=sr) if rate != sr else audio
    if wav.shape[0] == 1:
        wav = np.repeat(wav, 2, axis=0)
    wav = torch.from_numpy(np.ascontiguousarray(wav[:2], dtype=np.float32))
    ref = wav.mean(0)
    mean, std = ref.mean(), ref.std() + 1e-8
    t0 = time.time()
    cb = None
    if on_progress:
        subs = getattr(model, "models", [model])
        seg = int(subs[0].samplerate * float(subs[0].segment))
        total = len(range(0, wav.shape[-1], int(0.75 * seg))) * len(subs)
        done = [0]

        def cb(d):
            if d.get("state") == "end":
                done[0] += 1
                on_progress("split", min(1.0, done[0] / total))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    with torch.no_grad():
        stems = apply_model(model, ((wav - mean) / std)[None], device=device,
                            split=True, overlap=0.25, progress=progress and not on_progress, callback=cb)[0]
    stems = (stems * std + mean).numpy()
    status(f"  split in {time.time() - t0:.0f}s")
    if parts == "all":                                  # every part, 16 kHz mono (for the arranger)
        out = {name: librosa.resample(stems[i].mean(0), orig_sr=sr, target_sr=SR).astype(np.float32)
               for i, name in enumerate(model.sources)}
        out["original"] = librosa.resample(wav.numpy().mean(0), orig_sr=sr, target_sr=SR).astype(np.float32)
        return out, SR, ""
    if parts == "auto":
        parts, reason = auto_parts(analyse_stems(stems, list(model.sources), sr))
        if parts is None:
            status(f"Auto: {reason}")
            return wav.numpy().mean(0), sr, "Auto picked: original audio (" + reason + ")"
        status(f"Auto: {reason}")
        desc = "Auto picked: " + " + ".join(PART_NAMES[p] for p in parts) + " (" + reason + ")"
    else:
        desc = "Kept: " + " + ".join(PART_NAMES[p] for p in parts)
    status(f"  keeping: {', '.join(parts)}")
    mix = sum(stems[model.sources.index(p)] for p in parts).mean(0)
    return mix, sr, desc


def clean_notes(notes, min_len=0.06, min_vel=20, max_poly=8):
    """Drop blips, ghost notes and pile-ups that come from non-piano sounds."""
    # Re-join notes that got chopped by holes where drum hits were removed:
    # same key, restarts right as the previous one ends, and noticeably softer.
    notes = sorted((dict(n) for n in notes), key=lambda n: n["onset_time"])
    last, merged = {}, []
    for n in notes:
        p = last.get(n["midi_note"])
        if p and n["onset_time"] - p["offset_time"] < 0.04 and n["velocity"] < 0.8 * p["velocity"]:
            p["offset_time"] = max(p["offset_time"], n["offset_time"])
            continue
        last[n["midi_note"]] = n
        merged.append(n)
    notes = [n for n in merged
             if n["offset_time"] - n["onset_time"] >= min_len and n["velocity"] >= min_vel]
    # Limit how many notes can start at (nearly) the same moment: keep the loudest
    notes.sort(key=lambda n: n["onset_time"])
    kept, i = [], 0
    while i < len(notes):
        j = i
        while j < len(notes) and notes[j]["onset_time"] - notes[i]["onset_time"] < 0.03:
            j += 1
        group = sorted(notes[i:j], key=lambda n: -n["velocity"])[:max_poly]
        kept.extend(group)
        i = j
    return sorted(kept, key=lambda n: n["onset_time"])


def output_path(name, mode):
    """A safe, unused file name in the output folder (never overwrites an earlier song)."""
    OUT.mkdir(parents=True, exist_ok=True)
    safe = "".join(c for c in name if c not in '<>:"/\\|?*' and ord(c) >= 32).strip(" .") or "song"
    if safe.split(".")[0].upper() in {"CON", "PRN", "AUX", "NUL", *(f"COM{i}" for i in range(10)),
                                      *(f"LPT{i}" for i in range(10))}:
        safe = "song " + safe  # Windows reserved device names
    if mode in ("pop", "auto"):
        safe += f" ({mode})"
    path, n = OUT / f"{safe}.mid", 2
    while path.exists():
        path, n = OUT / f"{safe} ({n}).mid", n + 1
    return path


def arrange_song(audio, rate, midi_path, status=print, progress=True, on_progress=None):
    """Auto mode: a full piano arrangement fitted to the player piano (see arranger.py, docs/CALIBRATION.md)."""
    import json as _json
    import arranger
    stems, _, _ = separate(audio, rate, "all", status, progress,
                           (lambda st, f: on_progress("split", f)) if on_progress else None)
    # the band's keys/guitar part, as the piano model hears it
    x = stems["other"] / (np.abs(stems["other"]).max() + 1e-9) * 0.9
    x = np.concatenate([np.zeros(SR // 2, dtype=np.float32), x])
    status("Transcribing the band's keys/guitar part...")
    model = piano_model()
    report = (lambda f: on_progress("transcribe", f * 0.85)) if on_progress else (lambda f: None)
    with quiet(), transcribe_progress(report):
        other = model.transcribe(x, None)
    song = arranger.Song(stems, other["output_dict"], status=status)
    if on_progress: on_progress("transcribe", 0.95)
    lock = None
    try:
        import piano_check
        lf = arranger.LOCKS / f"{piano_check.base_name(Path(midi_path).name)}.intro.json"
        if lf.exists():
            lock = _json.loads(lf.read_text(encoding="utf-8")); status(f"Using the locked intro ({lf.name})")
    except Exception:
        pass
    notes, ped = arranger.arrange(song, arranger.Piano(), status=status, lock=lock)
    arranger.write_midi(midi_path, notes, ped)
    n_mel = sum(1 for n in notes if n["role"] == "melody")
    return (f"{len(notes)} notes arranged for your piano ({n_mel} melody, {len(song.choruses)} choruses, "
            f"{60 / song.bl:.0f} bpm)")


def convert(audio, rate, mode, parts, midi_path, status=print, progress=True, on_progress=None):
    """audio: [channels, samples] float array. Writes midi_path, returns a summary string."""
    import librosa
    from piano_transcription_inference.utilities import write_events_to_midi

    status(f"Mode: {MODE_NAMES[mode]}")
    desc = ""
    try:   # remember what the original sounds like, so a later piano check can compare against it
        import librosa
        import piano_check
        piano_check.save_reference(librosa.resample(np.asarray(audio.mean(0), dtype=np.float32), orig_sr=rate,
                                                    target_sr=piano_check.SR), Path(midi_path).name)
    except Exception as e:
        status(f"(couldn't save the song reference for piano checks: {e})")
    if mode == "auto":
        return arrange_song(audio, rate, midi_path, status, progress, on_progress)
    if mode == "pop":
        mono, rate, desc = separate(audio, rate, "auto" if mode == "auto" else parts, status, progress, on_progress)
    else:
        mono = audio.mean(0)
    mono = librosa.resample(np.asarray(mono, dtype=np.float32), orig_sr=rate, target_sr=SR)

    # Normalise loudness so quiet recordings still trigger notes
    peak = np.abs(mono).max()
    if peak > 0:
        mono = mono / peak * 0.9
    # Short silent lead-in so a note right at the start isn't missed
    mono = np.concatenate([np.zeros(SR // 2, dtype=mono.dtype), mono])

    status(f"Transcribing {len(mono) / SR:.0f}s of audio (this takes a little while on CPU)...")
    model = piano_model()
    t0 = time.time()
    report = (lambda f: on_progress("transcribe", f)) if on_progress else (lambda f: None)
    with quiet(), transcribe_progress(report):
        result = model.transcribe(mono, str(midi_path))
    notes = result["est_note_events"]
    msg = f"{len(notes)} notes"
    if mode in ("pop", "auto"):
        cleaned = clean_notes(list(notes))
        msg = f"{len(cleaned)} notes (removed {len(notes) - len(cleaned)} stray ones)"
        write_events_to_midi(start_time=0, note_events=cleaned,
                             pedal_events=result["est_pedal_events"], midi_path=str(midi_path))
    status(f"Done in {time.time() - t0:.0f}s - {msg}")
    return f"{msg}. {desc}" if desc else msg


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("file", nargs="?", help="audio file (mp3/wav/flac/...) - omit to record from the mic")
    ap.add_argument("--mode", choices=("auto", "piano", "pop"), default="piano")
    ap.add_argument("--parts", default="vocals,other,bass",
                    help="pop mode: parts to turn into piano (vocals, other, bass, drums)")
    ap.add_argument("--seconds", type=float, help="record for this many seconds")
    ap.add_argument("--name", help="output file name (without .mid)")
    args = ap.parse_args()

    parts = [p.strip() for p in args.parts.split(",") if p.strip()]
    bad = [p for p in parts if p not in PARTS]
    if bad or not parts:
        sys.exit(f"--parts must be from: {', '.join(PARTS)}")

    if args.file:
        src = Path(args.file)
        if not src.exists():
            sys.exit(f"Can't find {src}")
        print(f"Loading {src.name}...")
        audio, rate = load_file(str(src))
        name = args.name or src.stem
    else:
        audio, rate = record(args.seconds)
        name = args.name or input("Name for this song (Enter for a timestamp): ").strip() \
            or datetime.now().strftime("recording-%Y%m%d-%H%M%S")

    midi_path = output_path(name, args.mode)
    convert(audio, rate, args.mode, parts, midi_path)
    print(f"Saved: {midi_path}")
    print("Drag it into Piano Jukebox -> My Music to play it.")
    if os.name == "nt":
        os.startfile(OUT)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile arranger.py
"""Piano arranger: turns any song into a piano arrangement for a player piano, on the first try.

This is everything learned from the calibration runs on 'Dancing in the Moonlight' (arranged v1..v18,
12 real-piano checks) - see docs/CALIBRATION.md for the why behind every rule. In short:

  melody   the singer's pitch track (not the piano model's reading of the voice), notes start on the
           real syllables and end when the singer stops, short wobbles/blips merged, notes the song
           doesn't contain dropped, small gaps joined so it sounds sung
  bass     the real bass-guitar line (walking/passing notes), one strike per pluck
  keys     the band's keys/guitar part (piano-model transcription of the instruments), judged against
           the INSTRUMENTS (not the vocal-heavy mix) so the chorus doesn't empty out, always kept at
           least 5 semitones under the tune
  groove   every beat has something: bass on 1 & 3, a chord on 2 & 4 where the band leaves a gap;
           crowded beats are trimmed
  sections keys-only intro/outro played as transcribed; choruses found automatically (repeated +
           loudest + sung) get a fuller part, the tune in octaves and bass octaves, and swell in/out;
           light instrumental breaks are thinned to their high 'twinkle' figure
  piano    every note fitted to the player piano (piano_profile.json): loud enough to sound, long
           enough for the hammer, time for keys to reset, soft notes sent early, weak keys firmer,
           never too many keys ringing, pedal changed at every tune note and bass note
"""
import json
import time
from pathlib import Path

import numpy as np

SR = 16000
HERE = Path(__file__).resolve().parent
PROFILE = HERE / "piano_profile.json"
LOCKS = HERE / "locks"
NAMES = ["C", "C#", "D", "Eb", "E", "F", "F#", "G", "Ab", "A", "Bb", "B"]


# ============================================================================ piano calibration
class Piano:
    def __init__(self, path=PROFILE):
        p = json.loads(Path(path).read_text(encoding="utf-8")) if Path(path).exists() else {}
        self.p = p
        self.vol = p.get("jukebox_volume", 0.6)
        self.floor = p.get("jukebox_softest", 35)
        self.L = dict(melody_verse=73, melody_chorus=75, melody_octave_double=60, melody_light_break=60, bass=56,
                      bass_chorus=60, chords=52, chords_chorus=58, keys_part=58, light_figure=52, under_melody_cap=52,
                      minimum=45)
        self.L.update({k: v for k, v in p.get("levels", {}).items() if not k.startswith("_")})
        self.weak = set(p.get("weak_keys", []))
        self.weak_min = p.get("weak_key_minimum", 58)
        self.weak_over = {int(k): v for k, v in p.get("weak_key_overrides", {}).items()}
        self.min_note = p.get("min_note_seconds", 0.15)
        self.reset = p.get("key_reset_seconds", 0.12)
        self.merge = p.get("merge_same_key_within", 0.2)
        self.mel_repeat = p.get("melody_repeat_within", 0.15)
        self.lag = p.get("soft_note_lag", [[45, 0.027], [60, 0.020]])
        self.max_ring = p.get("max_keys_ringing", 12)
        pd = p.get("pedal", {})
        self.ped_min, self.ped_max = pd.get("min_press", 0.12), pd.get("max_span", 4.0)

    def vel(self, level):
        """MIDI velocity that lands at `level` on the piano (after the jukebox's volume)."""
        return int(np.clip(np.ceil(level / self.vol), 1, 127))

    def played(self, v):
        return max(self.floor, min(127, round(v * self.vol)))

    def key_min(self, p):
        return self.weak_over.get(p, self.weak_min) if p in self.weak else self.L["minimum"]

    def lag_for(self, v):
        pv = self.played(v)
        for upto, sec in self.lag:
            if pv <= upto: return sec
        return 0.0


# ============================================================================ analysis
def _merge_same_key(notes, gap):
    notes = sorted(notes, key=lambda n: n["s"]); last, out = {}, []
    for n in notes:
        p = last.get(n["p"])
        if p is not None and n["s"] - p["e"] < gap:
            p["e"] = max(p["e"], n["e"]); p["v"] = max(p["v"], n["v"]); continue
        n = dict(n); last[n["p"]] = n; out.append(n)
    return out


def _pitch_line(f0, voiced, rms, hop, smooth, min_len, jump, legato, lo, hi):
    """A line of notes from a pitch track: round to keys, median-smooth vibrato, new note only on a real move."""
    import librosa
    import scipy.ndimage as ndi
    midi = np.where(voiced & (f0 > 0), librosa.hz_to_midi(np.where(f0 > 0, f0, 1)), np.nan)
    k = max(1, int(smooth / hop)) | 1
    med = ndi.median_filter(np.where(np.isnan(midi), -1, midi), size=k, mode="nearest")
    med = np.where(ndi.median_filter((~np.isnan(midi)).astype(float), size=k) > 0.5, med, -1)
    notes, cur = [], None
    for i, m in enumerate(med):
        if cur and m > 0 and abs(m - cur["p"]) < jump:
            cur["e"] = (i + 1) * hop; cur["w"].append(rms[min(i, len(rms) - 1)]); continue
        if cur: notes.append(cur); cur = None
        if m > 0: cur = dict(s=i * hop, e=(i + 1) * hop, p=int(round(m)), w=[rms[min(i, len(rms) - 1)]])
    if cur: notes.append(cur)
    notes = [n for n in notes if n["e"] - n["s"] >= min_len and lo <= n["p"] <= hi]
    for n in notes: n["v"] = float(np.mean(n.pop("w")))
    notes = _merge_same_key(notes, 0.08)
    for a, b in zip(notes, notes[1:]):
        if 0 < b["s"] - a["e"] < legato: a["e"] = b["s"]
    return notes


class Song:
    """Everything the arranger needs to know about the song, computed once."""

    def __init__(self, stems, other_out, status=print):
        import librosa
        from piano_transcription_inference.utilities import RegressionPostProcessor
        t0 = time.time()
        self.stems = stems
        self.dur = len(stems["original"]) / SR
        y = stems["original"]
        status("Finding the beat...")
        _, beats = librosa.beat.beat_track(y=y, sr=SR, hop_length=512, units="time")
        self.beats = np.asarray(beats, float)
        if len(self.beats) < 8:                           # no clear beat: a steady grid at 100 bpm
            self.beats = np.arange(0, self.dur, 0.6)
        self.bl = float(np.median(np.diff(self.beats)))
        # the band's keys/guitar part, as the piano model heard it (with its own pedal)
        pp = RegressionPostProcessor(100, 88, 0.2, 0.3, 0.05, 0.2)
        notes, pedals = pp.output_dict_to_midi_events(other_out)
        self.keys_raw = [dict(s=n["onset_time"] - 0.5, e=n["offset_time"] - 0.5, p=int(n["midi_note"]), v=int(n["velocity"]))
                         for n in notes if n["offset_time"] - 0.5 > 0]
        self.keys_pedal = [(p["onset_time"] - 0.5, p["offset_time"] - 0.5) for p in (pedals or [])]
        status("Following the singer...")
        v = stems["vocals"]
        f0, vc, _ = librosa.pyin(v, fmin=80, fmax=1000, sr=SR, hop_length=160, frame_length=1024)
        self.f0, self.vc = np.nan_to_num(f0), vc
        self.vrms = librosa.feature.rms(y=v, frame_length=1024, hop_length=160)[0]
        env = librosa.onset.onset_strength(y=v, sr=SR, hop_length=160)
        self.sylls = librosa.onset.onset_detect(onset_envelope=env, sr=SR, hop_length=160, units="time", backtrack=True, delta=0.2, wait=8)
        n = min(len(self.vrms), len(self.vc))
        self.singing = (self.vrms[:n] > 0.12 * np.percentile(self.vrms, 95)) & self.vc[:n]
        status("Following the bass...")
        bf0, bvc, _ = librosa.pyin(stems["bass"], fmin=30, fmax=300, sr=SR, hop_length=320, frame_length=4096)
        self.bf0, self.bvc = np.nan_to_num(bf0), bvc
        self.brms = librosa.feature.rms(y=stems["bass"], frame_length=1024, hop_length=320)[0]
        benv = librosa.onset.onset_strength(y=stems["bass"], sr=SR, hop_length=160)
        self.bass_on = librosa.onset.onset_detect(onset_envelope=benv, sr=SR, hop_length=160, units="time", delta=0.07, wait=5)
        status("Reading the harmony...")
        self.song_pc = self._pc(stems["vocals"] + stems["other"] + stems["bass"])       # what the song sounds like
        self.inst_pc = self._pc(stems["other"] + 0.5 * stems["bass"])                  # what the INSTRUMENTS play
        fr = librosa.time_to_frames(self.beats, sr=SR, hop_length=512)
        self.beat_chroma = librosa.util.sync(librosa.feature.chroma_cqt(y=stems["other"] + stems["bass"], sr=SR, hop_length=512), fr, aggregate=np.mean)[:, 1:len(self.beats)]
        self.beat_bass = librosa.util.sync(librosa.feature.chroma_cqt(y=stems["bass"], sr=SR, hop_length=512), fr, aggregate=np.mean)[:, 1:len(self.beats)]
        self._sections(fr)
        status(f"  analysed in {time.time() - t0:.0f}s: {60 / self.bl:.0f} bpm, {len(self.choruses)} choruses, "
               f"{len(self.light)} light break(s), keys-only intro {self.intro_end:.1f}s")

    def _pc(self, y, hop=0.1):
        import librosa
        import scipy.ndimage as ndi
        C = np.abs(librosa.cqt(y, sr=SR, hop_length=int(hop * SR), fmin=librosa.midi_to_hz(21) / 2 ** (1 / 36), n_bins=264, bins_per_octave=36)) ** 2
        C = C.reshape(88, 3, -1).max(1); pc = np.zeros((12, C.shape[1]))
        for k in range(88): pc[(k + 21) % 12] += C[k]
        loud = pc.sum(0) > 0.02 * np.percentile(pc.sum(0), 95)
        return ndi.median_filter(pc / (pc.max(0, keepdims=True) + 1e-12), size=(1, 3)), loud

    def strength(self, n, which="song", span=0.3):
        pc, _ = self.song_pc if which == "song" else self.inst_pc
        a = int(n["s"] / 0.1); b = max(a + 1, int(min(n["e"], n["s"] + span) / 0.1))
        a, b = min(max(a, 0), pc.shape[1] - 1), min(b, pc.shape[1])
        return float(pc[n["p"] % 12, a:b].mean()) if b > a else 1.0

    def _sections(self, fr):
        import librosa
        s, B = self.stems, self.beats
        lvl = lambda y: (lambda r: librosa.util.sync(r[None], fr, aggregate=np.mean)[0][1:len(B)] / (np.percentile(r, 95) + 1e-9))(
            librosa.feature.rms(y=y, frame_length=2048, hop_length=512)[0])
        vo, ba, dr, ot = lvl(s["vocals"]), lvl(s["bass"]), lvl(s["drums"]), lvl(s["other"])
        nb = len(vo)
        # keys-only intro (and outro): no singing, bass or drums
        keys_only = (vo < 0.15) & (ba < 0.15) & (dr < 0.15)
        i = 0
        while i < nb and keys_only[i]: i += 1
        self.intro_end = float(B[i]) if i >= 6 else 0.0
        j = nb - 1
        while j > 0 and keys_only[j]: j -= 1
        self.outro_keys = float(B[j + 1]) if nb - 1 - j >= 6 else self.dur + 1
        # choruses: 2-bar patterns that repeat elsewhere + loudest + sung
        C = self.beat_chroma / (np.linalg.norm(self.beat_chroma, axis=0) + 1e-9)
        M = np.zeros((12, nb)); hop = 0.01
        for k in range(nb):
            a, b = int(B[k] / hop), int(B[k + 1] / hop)
            seg = self.f0[a:b][self.vc[a:b] & (self.f0[a:b] > 0)]
            if len(seg): M[np.round(librosa.hz_to_midi(seg)).astype(int) % 12, k] += 1
        F = np.vstack([C, M / (np.linalg.norm(M, axis=0) + 1e-9)])
        W = 8; n = F.shape[1] - W
        self.choruses = []
        if n > 40:
            X = np.stack([F[:, k:k + W].ravel() for k in range(n)]); X /= np.linalg.norm(X, axis=1, keepdims=True) + 1e-9
            S = X @ X.T
            rep = np.array([np.sort(S[k][np.abs(np.arange(n) - k) > 16])[-3:].mean() for k in range(n)])
            loud = np.array([(vo + ba + ot)[k:k + W].mean() for k in range(n)]); voc = np.array([vo[k:k + W].mean() for k in range(n)])
            z = lambda a: (a - a.mean()) / (a.std() + 1e-9)
            score = z(rep) + z(loud) + 0.5 * z(voc); on = score > np.percentile(score, 70)
            k = 0
            while k < n:
                if on[k]:
                    m = k
                    while m < n and on[m]: m += 1
                    if m - k >= 6: self.choruses.append([float(B[k]), float(B[min(m + W - 1, len(B) - 1)])])
                    k = m
                else: k += 1
            merged = []
            for a, b in self.choruses:                    # join overlapping chorus windows
                if merged and a <= merged[-1][1] + 2 * self.bl: merged[-1][1] = max(merged[-1][1], b)
                else: merged.append([a, b])
            self.choruses = [tuple(c) for c in merged]
        # light instrumental breaks: barely any singing or bass, instruments still going (>= 3 bars)
        import scipy.ndimage as ndi
        light = (vo < 0.35) & (ba < 0.5) & (ot > 0.3) & ~keys_only
        light = ndi.median_filter(light.astype(float), size=5) > 0.5          # ignore single-beat blips
        self.light, k = [], 0
        while k < nb:
            if light[k] and B[k] > self.intro_end + 4:
                m = k
                while m < nb and light[m]: m += 1
                if m - k >= 8: self.light.append((float(B[k]), float(B[min(m, len(B) - 1)])))
                k = m
            else: k += 1
        last_ch = max([b for a, b in self.choruses], default=self.dur - 20)
        self.outro = max(last_ch, self.dur - 15)
        # which beats are 1 & 3 (bass + kick), and where bars start
        import librosa as lr
        low = lr.feature.rms(y=s["bass"] + lr.effects.preemphasis(s["drums"], coef=-0.9), hop_length=160)[0]
        bi = [min(len(low) - 1, int(b * 100)) for b in B]
        par = int(np.argmax([np.mean([low[bi[k]] for k in range(p, len(B), 2)]) for p in (0, 1)]))
        self.first_strong = next((k for k in range(len(B)) if B[k] >= self.intro_end + 0.05 and k % 2 == par), 0)
        bp = int(np.argmax([np.mean([low[bi[k]] for k in range(self.first_strong + 2 * q, len(B), 4)] or [0]) for q in (0, 1)]))
        self.first_bar = self.first_strong + 2 * bp

    def in_chorus(self, t): return any(a <= t < b for a, b in self.choruses)
    def in_light(self, t): return any(a <= t < b for a, b in self.light)


# ============================================================================ the parts
def melody(S, P):
    """The singer's tune: syllable-accurate starts, ends when the singer stops, no blips or unsupported notes."""
    import librosa
    import scipy.ndimage as ndi
    hop = 0.01; n = min(len(S.f0), len(S.singing))
    midi = np.where(S.vc[:n] & (S.f0[:n] > 0), librosa.hz_to_midi(np.where(S.f0[:n] > 0, S.f0[:n], 1)), np.nan)
    k = int(0.1 / hop) | 1
    med = ndi.median_filter(np.where(np.isnan(midi), -1, midi), size=k, mode="nearest")
    on = S.singing[:n] & (ndi.median_filter((~np.isnan(midi)).astype(float), size=k) > 0.5)
    notes, cur = [], None
    for i in range(n):
        m = med[i] if on[i] and med[i] > 0 else None
        if cur and m is not None and abs(m - cur["p"]) < 0.75:
            cur["e"] = (i + 1) * hop; cur["w"].append(S.vrms[i]); continue
        if cur: notes.append(cur); cur = None
        if m is not None: cur = dict(s=i * hop, e=(i + 1) * hop, p=int(round(m)), w=[S.vrms[i]])
    if cur: notes.append(cur)
    notes = [x for x in notes if x["e"] - x["s"] >= 0.05 and 48 <= x["p"] <= 84]
    for x in notes:                                     # start on the real syllable (consonants come before pitch)
        j = np.searchsorted(S.sylls, x["s"] + 0.03) - 1
        if j >= 0 and x["s"] - 0.15 <= S.sylls[j] <= x["s"] + 0.03: x["s"] = float(S.sylls[j])
    out = []
    for x in notes:                                     # same pitch, new clear syllable -> strike again
        s = x["s"]
        for c in S.sylls[np.searchsorted(S.sylls, x["s"] + 0.12): np.searchsorted(S.sylls, x["e"] - 0.12)]:
            out.append(dict(x, s=s, e=c - 0.02)); s = float(c)
        out.append(dict(x, s=s))
    notes = sorted(out, key=lambda x: x["s"])
    for a, b in zip(notes, notes[1:]):
        if a["e"] > b["s"]: a["e"] = b["s"] - 0.01
    for x in notes: x.pop("w", None)
    notes = [x for x in notes if x["e"] - x["s"] > 0.04]
    ch = True                                           # merge one-off pitch blips into a neighbour
    while ch:
        ch = False
        for i in range(1, len(notes) - 1):
            x, a, b = notes[i], notes[i - 1], notes[i + 1]
            if x["e"] - x["s"] < 0.1 and abs(x["p"] - a["p"]) > 2 and abs(x["p"] - b["p"]) > 2 and x["s"] - a["e"] < 0.05:
                a["e"] = x["e"]; del notes[i]; ch = True; break
    kept = []                                           # notes the song doesn't contain at all: move or drop
    for x in notes:
        if x["s"] < S.intro_end: continue
        if S.strength(x) >= 0.15: kept.append(x); continue
        best = next((x["p"] + d for d in (1, -1, 2, -2) if S.strength(dict(x, p=x["p"] + d)) >= 0.4), None)
        if best is not None: kept.append(dict(x, p=int(best)))
        elif S.strength(x) < 0.06 and x["e"] - x["s"] < 0.25: continue
        else: kept.append(x)
    late = S.dur * 0.85
    for a, b in zip(kept, kept[1:]):                    # join small gaps so it sounds sung
        gap = b["s"] - a["e"]
        if 0 < gap <= (0.30 if a["s"] >= late else 0.20): a["e"] = b["s"] - 0.02
    for x in kept:
        x["role"] = "melody"
        lvl = P.L["melody_light_break"] if S.in_light(x["s"]) else (P.L["melody_chorus"] if S.in_chorus(x["s"]) else P.L["melody_verse"])
        x["v"] = P.vel(lvl)
    return kept


def bass_line(S, P):
    line = _pitch_line(S.bf0, S.bvc, S.brms, 0.02, 0.08, 0.06, 0.7, 0.15, 28, 55)
    ons = np.sort(S.bass_on); out = []
    for x in line:                                      # one strike per pluck
        while x["p"] < 31: x["p"] += 12
        s = x["s"]
        for c in ons[np.searchsorted(ons, x["s"] + 0.07): np.searchsorted(ons, x["e"] - 0.07)]:
            if c - s >= 0.07: out.append(dict(x, s=s, e=c - 0.015)); s = c
        out.append(dict(x, s=s))
    return [dict(x, role="bass", v=P.vel(P.L["bass_chorus"] if S.in_chorus(x["s"]) else P.L["bass"]))
            for x in out if x["s"] >= S.intro_end and S.strength(x) >= 0.08]


def keys_part(S, P, mel):
    """The band's keys/guitar part under the tune: only what the instruments really play, >= 5 semitones below."""
    ms = sorted(mel, key=lambda m: m["s"]); st = np.array([m["s"] for m in ms]); out = []
    for x in S.keys_raw:
        if x["s"] < S.intro_end or x["e"] - x["s"] < 0.05 or S.in_light(x["s"]): continue
        if S.strength(x, "inst") < 0.35: continue
        k = np.searchsorted(st, x["s"] + 0.1, "right") - 1
        mp = ms[k]["p"] if 0 <= k < len(ms) and ms[k]["s"] - 0.1 <= x["s"] < ms[k]["e"] + 0.05 else None
        p = x["p"]; top = (mp - 5) if mp else 72
        while p > top and p - 12 >= 43: p -= 12
        while p < 46: p += 12
        if p > top: continue
        out.append(dict(s=x["s"], e=x["e"], p=int(p), role="keys",
                        v=P.vel(P.L["keys_part"] if S.in_chorus(x["s"]) else P.L["chords"])))
    return out


def harmony(S, i):
    c = S.beat_chroma[:, i:i + 2].mean(1); b = S.beat_bass[:, i:i + 2].mean(1)
    root = int(np.argmax(b)) if b.max() > 1.5 * (b.mean() + 1e-9) else int(np.argmax(c))
    pcs = [int(p) for p in np.argsort(-c) if c[p] >= 0.45 * c.max()][:4]
    if root not in pcs: pcs = [root] + pcs[:3]
    t0, t1 = S.beats[i], S.beats[min(len(S.beats) - 1, i + 2)]
    return root, [p for p in pcs if p == root or S.strength(dict(s=t0, e=t1, p=p), span=t1 - t0) >= 0.2] or [root]


def _voice(pcs, top, prev):
    centre = float(np.mean(prev)) if prev else 54
    for lo in (46, 41):
        out = sorted({min([p for p in range(lo, top + 1) if p % 12 == pc], key=lambda p: abs(p - centre))
                      for pc in pcs if any(p % 12 == pc for p in range(lo, top + 1))})
        if len(out) >= 2 or lo == 41: break
    return out or [40 + (pcs[0] - 40) % 12]


def groove_and_sections(S, P, mel, parts):
    """Steady pulse (fill empty beats), chorus octaves, light-break figure, final bars."""
    B, bl = S.beats, S.bl
    ms = sorted(mel, key=lambda m: m["s"]); st = np.array([m["s"] for m in ms])
    def mel_low(t, d):
        lo = None
        for m in ms[max(0, np.searchsorted(st, t - 2)): np.searchsorted(st, t + d)]:
            if m["s"] < t + d and m["e"] > t: lo = m["p"] if lo is None else min(lo, m["p"])
        return lo
    beat_of = lambda t: int(np.searchsorted(B, t + 0.08) - 1)
    per = {}
    for x in parts:
        per.setdefault(beat_of(x["s"]), []).append(x)
    keep = []
    for k, L in per.items():                            # crowded beats: bass first, then loudest; max 4
        keep += sorted(L, key=lambda x: (x["p"] >= 48, -x["v"]))[:4]
    out = keep; prev = [52, 55, 59]; end_groove = S.dur - 10
    for i in range(S.first_strong, len(B) - 1):
        t = B[i]; nxt = B[i + 1]
        if t >= min(end_groove, S.outro_keys): break
        pos = (i - S.first_bar) % 4; root, pcs = harmony(S, i - (pos % 2)); chorus = S.in_chorus(t)
        if S.in_light(t):
            if pos == 0: out.append(dict(s=t, e=t + 4 * bl - 0.12, p=36 + (root - 36) % 12, role="bass", v=P.vel(46)))
            continue
        if per.get(i): continue
        if pos in (0, 2):
            out.append(dict(s=t, e=nxt + bl - 0.12, p=36 + (root - 36) % 12, role="bass", v=P.vel(P.L["bass_chorus"] if chorus else P.L["bass"])))
        else:
            ml = mel_low(t, bl); vs = _voice(pcs, min((ml - 5) if ml else 64, 64), prev); prev = vs
            out += [dict(s=t + 0.01, e=nxt - 0.12, p=p, role="chord", v=P.vel(P.L["chords_chorus"] if chorus else P.L["chords"])) for p in vs]
    for i in range(S.first_bar, len(B) - 1, 4):         # chorus: bass octave on each bar's first beat
        t = B[i]
        if not S.in_chorus(t): continue
        bs = [x for x in out if x["role"] == "bass" and abs(x["s"] - t) < 0.12]
        if bs and bs[0]["p"] - 12 >= 28: out.append(dict(bs[0], p=bs[0]["p"] - 12, role="bass"))
    for m in mel:                                       # chorus: the tune in octaves (held notes)
        if S.in_chorus(m["s"]) and m["p"] + 12 <= 96 and m["e"] - m["s"] >= 0.25:
            out.append(dict(s=m["s"] + 0.005, e=m["e"], p=m["p"] + 12, role="double", v=P.vel(P.L["melody_octave_double"])))
    for a, b in S.light:                                # light break: just the high twinkle figure
        fig = [x for x in S.keys_raw if a <= x["s"] < b and x["p"] >= 84 and x["e"] - x["s"] >= 0.05]
        if not fig: fig = [x for x in S.keys_raw if a <= x["s"] < b and x["p"] >= 72 and x["e"] - x["s"] >= 0.05]
        out += [dict(s=x["s"], e=max(x["e"], x["s"] + 0.15), p=x["p"], role="figure", v=P.vel(P.L["light_figure"])) for x in fig]
    for i in range(len(B) - 1):                         # final bars: one held chord per bar
        t = B[i]
        if t < end_groove or (i - S.first_bar) % 4 or t >= S.outro_keys: continue
        root, pcs = harmony(S, i); prev = _voice(pcs, 64, prev)
        out.append(dict(s=t, e=t + 4 * bl - 0.12, p=36 + (root - 36) % 12, role="bass", v=P.vel(50)))
        out += [dict(s=t + 0.02, e=t + 4 * bl - 0.12, p=p, role="chord", v=P.vel(48)) for p in prev]
    return out


def keys_only_parts(S, P):
    """Keys-only intro/outro: play the band's keys part as transcribed (that's what sounded beautiful)."""
    out, ped = [], []
    for a, b in ((0.0, S.intro_end), (S.outro_keys, S.dur + 1)):
        if b - a < 3: continue
        for x in S.keys_raw:
            if a <= x["s"] < b: out.append(dict(x, role="intro", v=max(x["v"], P.vel(P.L["minimum"]))))
        ped += [(max(a, s), min(b, e)) for s, e in S.keys_pedal if e > a and s < b and min(b, e) - max(a, s) > 0.1]
    return out, ped


# ============================================================================ fitting it to the piano
def dynamics(S, P, notes):
    """Smooth section levels: fade the groove in after the intro, swell into choruses, settle after, taper the end."""
    bar = 4 * S.bl; t0 = S.beats[S.first_strong] if len(S.beats) else 0
    def level(t):
        L = 1.0
        for a, b in S.choruses:
            if a - bar <= t < a: L = max(L, 1.0 + 0.12 * (t - (a - bar)) / bar)
            elif b <= t < b + bar: L = max(L, 1.12 - 0.12 * (t - b) / bar)
        if S.in_light(t): L = 0.75
        if t < t0 + 2 * bar: L *= 0.7 + 0.3 * max(0.0, (t - t0) / (2 * bar))
        if t >= S.dur - 12: L *= max(0.6, 1.0 - 0.4 * (t - (S.dur - 12)) / 12)
        return L
    for x in notes:
        if x["role"] in ("melody", "intro", "double", "figure"): continue
        x["v"] = P.vel(min(76, P.played(x["v"]) * level(x["s"])))


def make_room(S, P, notes):
    """Nothing tangled just under the tune; backing struck with a tune note follows it (melody lead) and softer."""
    mel = sorted([x for x in notes if x["role"] == "melody"], key=lambda m: m["s"]); st = np.array([m["s"] for m in mel])
    def at(t):
        k = np.searchsorted(st, t + 0.1, "right") - 1
        for j in (k, k - 1):
            if 0 <= j < len(mel) and mel[j]["s"] - 0.1 <= t < mel[j]["e"] + 0.05: return mel[j]
    out = []
    for x in notes:
        if x["role"] in ("melody", "intro", "double", "bass", "figure") or x["p"] < 48: out.append(x); continue
        m = at(x["s"])
        if m is not None:
            if x["p"] >= m["p"]: continue
            if m["p"] - x["p"] < 5:
                if x["p"] - 12 >= 43: x["p"] -= 12
                else: continue
            if abs(m["s"] - x["s"]) <= 0.06 and m["p"] - x["p"] <= 12:
                d = m["s"] + 0.035 - x["s"]; x["s"] += d; x["e"] += d
                x["v"] = min(x["v"], P.vel(P.L["under_melody_cap"]))
        out.append(x)
    return out


def fit_to_piano(P, notes, intro_end):
    """Loud enough to sound, long enough for the hammer, keys get time to reset, soft notes sent early."""
    for x in notes:
        if x["role"] != "melody": x["v"] = max(x["v"], P.vel(P.key_min(x["p"])))
        lag = P.lag_for(x["v"]); x["s"] = max(0.0, x["s"] - lag); x["e"] -= lag
    by = {}
    for x in sorted(notes, key=lambda x: x["s"]): by.setdefault(x["p"], []).append(x)
    out = []
    for p, seq in by.items():
        keep = []
        for x in seq:
            lim = P.mel_repeat if (x["role"] == "melody" and keep and keep[-1]["role"] == "melody") else P.merge
            if keep and x["s"] - keep[-1]["s"] < lim and x["s"] >= intro_end:
                keep[-1]["e"] = max(keep[-1]["e"], x["e"]); keep[-1]["v"] = max(keep[-1]["v"], x["v"]); continue
            keep.append(x)
        for i, x in enumerate(keep):
            nxt = keep[i + 1]["s"] if i + 1 < len(keep) else 1e9
            if x["s"] >= intro_end: x["e"] = max(x["e"], x["s"] + P.min_note)
            x["e"] = max(min(x["e"], nxt - P.reset), x["s"] + 0.03)
        out += keep
    mel = sorted([x for x in out if x["role"] == "melody"], key=lambda x: x["s"])
    for a, b in zip(mel, mel[1:]):                      # the tune stays one note at a time
        if a["e"] > b["s"] - 0.01: a["e"] = max(a["s"] + 0.08, b["s"] - 0.01)
    return out


def pedal(S, P, notes, intro_ped):
    t0 = S.intro_end
    ped = [(a, min(b, t0 - 0.02)) for a, b in intro_ped if a < t0 and min(b, t0 - 0.02) - a > 0.1]
    mel = [x for x in notes if x["role"] == "melody"]
    pts = sorted({round(x["s"], 3) for x in mel} | {round(x["e"], 3) for x in mel} |
                 {round(x["s"], 3) for x in notes if x["role"] == "bass"} | {round(t0, 3), round(S.dur, 3)})
    ped += [(a + 0.05, b - 0.02) for a, b in zip(pts, pts[1:])
            if a >= t0 and (b - 0.02) - (a + 0.05) >= P.ped_min and b - a <= P.ped_max]
    ped += [(a, b) for a, b in intro_ped if a >= S.outro_keys]
    return sorted(ped)


def cap_ringing(P, notes, ped, start):
    """Never more than max_keys_ringing at once: drop the softest keys-part notes first."""
    def rings(x):
        for a, b in ped:
            if a <= x["e"] < b: return b
        return x["e"]
    ends = {id(x): rings(x) for x in notes}
    dropped = 0
    for t in np.arange(start, max([x["e"] for x in notes] + [start]), 0.25):
        on = [x for x in notes if x["s"] <= t < ends[id(x)]]
        extra = len(on) - P.max_ring
        if extra <= 0: continue
        for x in sorted([x for x in on if x["role"] == "keys"], key=lambda x: x["v"])[:extra]:
            notes.remove(x); dropped += 1
    return dropped


# ============================================================================ entry point
def arrange(S, P=None, status=print, lock=None):
    P = P or Piano()
    mel = melody(S, P)
    intro, intro_ped = keys_only_parts(S, P)
    if lock:                                            # a hand-approved intro (locks/<song>.intro.json)
        intro = [dict(s=n["s"], e=n["e"], p=n["p"], v=n["v"], role="intro") for n in lock["notes"]]
        intro_ped = [tuple(p) for p in lock.get("pedal", [])]
        S.intro_end = max(S.intro_end, lock.get("until", 0))
        mel = [m for m in mel if m["s"] >= S.intro_end]
    bass = bass_line(S, P); keys = keys_part(S, P, mel)
    backing = groove_and_sections(S, P, mel, bass + keys)
    notes = intro + mel + backing
    dynamics(S, P, notes)
    notes = make_room(S, P, notes)
    notes = fit_to_piano(P, notes, S.intro_end)
    ped = pedal(S, P, notes, intro_ped)
    dropped = cap_ringing(P, notes, ped, S.intro_end)
    roles = {}
    for x in notes: roles[x["role"]] = roles.get(x["role"], 0) + 1
    status(f"Arranged: {len(notes)} notes ({', '.join(f'{k} {v}' for k, v in sorted(roles.items()))}), "
           f"{len(ped)} pedal presses{f', {dropped} dropped to avoid jumble' if dropped else ''}")
    return notes, ped


def write_midi(path, notes, ped):
    import mido
    mf = mido.MidiFile(ticks_per_beat=480); tr = mido.MidiTrack(); mf.tracks.append(tr)
    tr.append(mido.MetaMessage("set_tempo", tempo=500000))
    ev = []
    for n in notes:
        p = int(np.clip(n["p"], 21, 108))
        ev.append((max(0.0, n["s"]), 1, mido.Message("note_on", note=p, velocity=int(np.clip(n["v"], 1, 127)))))
        ev.append((max(0.0, n["e"]), 0, mido.Message("note_off", note=p, velocity=0)))
    for a, b in ped:
        ev.append((max(0.0, a), 2, mido.Message("control_change", control=64, value=127)))
        ev.append((max(0.0, b), -1, mido.Message("control_change", control=64, value=0)))
    ev = sorted(((int(round(t * 960)), pr, m) for t, pr, m in ev), key=lambda x: (x[0], x[1])); last = 0
    for tick, _, m in ev:
        tr.append(m.copy(time=tick - last)); last = tick
    mf.save(str(path))

In [ ]:
%%writefile piano_check.py
"""Piano check: record the real piano playing a song and find where it goes wrong.

  python piano_check.py --midi "output/Song.mid" --seconds 60      record 60 s from the mic now
  python piano_check.py --midi "output/Song.mid" --file take.wav   analyse an existing recording

The recording is lined up with the MIDI automatically (you can start recording at any point
in the song). Then, every few seconds, it checks:
  - timing:   does the piano strike the notes when the MIDI says? (early / late / missing)
  - notes:    does what you hear match the notes that were sent?
  - melody:   does the tune stand out over the backing, on the real instrument?
  - song:     does it sound like the original song there? (if the song was converted here)
and writes a report with timestamps to output/checks/.
"""
import argparse
import json
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np

SR = 16000
HOP = 160                      # 10 ms
HERE = Path(__file__).resolve().parent
OUT = HERE / "output"
CHECKS = OUT / "checks"
REFS = OUT / "refs"
WINDOW = 5.0                   # seconds per checked section


# ------------------------------------------------------------------------------ MIDI
def read_midi(data):
    import io
    import mido
    mf = mido.MidiFile(file=io.BytesIO(data)) if isinstance(data, (bytes, bytearray)) else mido.MidiFile(str(data))
    t, on, notes, ped, down = 0.0, {}, [], [], None
    for m in mf:
        t += m.time
        if m.type == "control_change" and m.control == 64:
            if m.value >= 64 and down is None: down = t
            elif m.value < 64 and down is not None: ped.append((down, t)); down = None
        elif m.type == "note_on" and m.velocity:
            on.setdefault(m.note, []).append((t, m.velocity))
        elif m.type in ("note_off", "note_on") and on.get(m.note):
            s, v = on[m.note].pop(0); notes.append(dict(s=s, e=t, p=m.note, v=v))
    if down is not None: ped.append((down, t))
    return notes, ped, mf.length


def sounding_until(n, ped):
    for s, e in ped:
        if s <= n["e"] < e: return e
    return n["e"]


def melody_line(notes, ped):
    """The tune a listener follows: the highest note sounding (from middle C up) at each moment."""
    mel = []
    for n in sorted(notes, key=lambda n: n["s"]):
        if n["p"] < 60: continue
        if mel and n["s"] < mel[-1]["e"] - 0.02:
            if n["p"] > mel[-1]["p"]: mel[-1] = dict(mel[-1], e=n["s"]); mel.append(n)
            continue
        mel.append(n)
    return [m for m in mel if m["e"] - m["s"] > 0.04]


# ------------------------------------------------------------------------------ audio features
def onset_env(y):
    import librosa
    return librosa.onset.onset_strength(y=y, sr=SR, hop_length=HOP)


def midi_onset_train(notes, n_frames):
    x = np.zeros(n_frames)
    for n in notes:
        i = int(n["s"] * SR / HOP)
        if 0 <= i < n_frames: x[i] += (n["v"] / 127) ** 1.5
    k = np.exp(-np.arange(8) / 2.5)                       # a struck note decays in the onset curve
    return np.convolve(x, k)[:n_frames]


def align(rec, notes, song_len, hint=None):
    """Where in the song does the recording start? Cross-correlate onset curves (FFT)."""
    env = onset_env(rec); env = (env - env.mean()) / (env.std() + 1e-9)
    n_song = int((song_len + 2) * SR / HOP)
    train = midi_onset_train(notes, n_song); train = (train - train.mean()) / (train.std() + 1e-9)
    L = len(train) + len(env)
    F = np.fft.rfft(train, L) * np.conj(np.fft.rfft(env, L))
    corr = np.fft.irfft(F, L)[: max(1, len(train) - len(env) // 4)]
    if hint is not None:
        lo, hi = max(0, int((hint - 3) * SR / HOP)), int((hint + 3) * SR / HOP)
        mask = np.full(len(corr), -np.inf); mask[lo:hi] = 0; corr = corr + mask
    k = int(np.argmax(corr))
    finite = corr[np.isfinite(corr)]
    conf = float((corr[k] - np.median(finite)) / (np.std(finite) + 1e-9))
    return k * HOP / SR, conf


def robust_align(rec, notes, song_len, hint=None):
    """Line up 30 s pieces across the whole recording and take the consensus offset, so a repetitive
    passage (e.g. an intro riff) can't pull the alignment a phrase off. Falls back to align()."""
    dur = len(rec) / SR
    if dur < 45:
        return align(rec, notes, song_len, hint)
    offs = []
    for c0 in np.arange(0, dur - 30, 15):
        a = int(c0 * SR)
        s, conf = align(rec[a: a + 30 * SR], notes, song_len)
        if conf >= 6: offs.append((s - c0, conf))
    if len(offs) < 2:
        return align(rec, notes, song_len, hint)
    o = np.array([x[0] for x in offs]); med = float(np.median(o))
    agree = np.abs(o - med) <= 0.1
    if agree.sum() < max(2, 0.5 * len(o)):
        return align(rec, notes, song_len, hint)
    # each piece is aligned to 10 ms; the consensus can also be slightly negative (recording began first)
    return float(np.median(o[agree])), float(np.mean([x[1] for x, ok in zip(offs, agree) if ok]))


def chroma(y):
    import librosa
    c = librosa.feature.chroma_cqt(y=y, sr=SR, hop_length=1024)
    return c / (np.linalg.norm(c, axis=0, keepdims=True) + 1e-9)


def roll_chroma(notes, ped, t0, n_frames):
    hop = 1024 / SR; c = np.zeros((12, n_frames))
    for n in notes:
        a = int((n["s"] - t0) / hop); b = int((sounding_until(n, ped) - t0) / hop) + 1
        a, b = max(0, a), min(n_frames, b)
        if b <= a: continue
        tt = (np.arange(a, b) * hop + t0 - n["s"]).clip(0)
        env = (n["v"] / 127) ** 1.7 * np.exp(-tt / np.interp(n["p"], [21, 108], [4.0, 0.6]))
        for k, w in ((0, 1.0), (0, 0.41), (7, 0.24), (0, 0.16), (4, 0.12)):
            c[(n["p"] + k) % 12, a:b] += w * env
    return c / (np.linalg.norm(c, axis=0, keepdims=True) + 1e-9)


def note_energy(rec):
    """Energy at each piano key over time: constant-Q with 3 bins per semitone (so neighbouring keys
    don't blur together), 10 ms grid, taking the strongest bin around each key's exact pitch."""
    import librosa
    C = np.abs(librosa.cqt(rec, sr=SR, hop_length=HOP, fmin=librosa.midi_to_hz(21) / 2 ** (1 / 36),
                           n_bins=88 * 3, bins_per_octave=36)) ** 2
    return C.reshape(88, 3, -1).max(1)                    # [88 keys, frames]


NOTE_NAMES = ["C", "C#", "D", "Eb", "E", "F", "F#", "G", "Ab", "A", "Bb", "B"]
def key_name(p):
    return f"{NOTE_NAMES[p % 12]}{p // 12 - 1}"


def partial_energy(E):
    """A key's evidence = its fundamental plus its first overtones (overtones are shorter-windowed in
    the constant-Q transform, so they pin down WHEN a note starts much more precisely)."""
    P = E.copy()
    for k, w in ((12, 0.6), (19, 0.4), (24, 0.3), (28, 0.2)):
        P[:-k] += w * E[k:]
    return P


T_HOP = 80                     # 5 ms grid for timing
T_FFT = 1024


def stft_mag(rec):
    """Two spectrograms on a 5 ms grid: a short window (32 ms, sharp timing) for notes from C3 up,
    and a longer one (64 ms, enough frequency detail) for the bass."""
    import librosa
    return {n: np.abs(librosa.stft(rec, n_fft=n, hop_length=T_HOP, center=True)) for n in (512, 1024)}


def harmonic_bins(p, nfft, lo=150, hi=7000):
    f0 = 440 * 2 ** ((p - 69) / 12); df = SR / nfft
    B = 0.0004 if p < 60 else 0.0008                            # piano strings are slightly inharmonic
    return {int(round(k * f0 * np.sqrt(1 + B * k * k) / df)) for k in range(1, 11) if lo <= k * f0 <= hi}


def note_flux(S, p, others=()):
    """How sharply this note's own harmonics jump, every 5 ms (spectral flux on just those bins).
    Only harmonics NO other nearby note shares are used, so it's about THIS note."""
    nfft = 512 if p >= 48 else 1024; X = S[nfft]
    taken = set()
    for q in others:
        for b in harmonic_bins(q, nfft, 60, 7600): taken |= {b - 2, b - 1, b, b + 1, b + 2}
    bins = sorted(b for b in harmonic_bins(p, nfft) if b not in taken)
    rows = [b for b in bins if 0 <= b < X.shape[0]]
    if len(rows) < 2: return None                                  # nothing unique to listen to
    M = X[rows]
    F = np.zeros_like(M); F[:, 2:] = np.maximum(0, M[:, 2:] - M[:, :-2])
    return F, M                                                    # per-harmonic flux, per-harmonic level


def leading_edge(w):
    """Where a jump starts: first frame (with sub-frame interpolation) reaching half the peak."""
    j = int(np.argmax(w)); e = j
    while e > 0 and w[e - 1] >= 0.5 * w[j]: e -= 1
    if e > 0:
        return e - 1 + float(np.clip((0.5 * w[j] - w[e - 1]) / (w[e] - w[e - 1] + 1e-9), 0, 1)), w[j]
    return float(e), w[j]


def note_by_note(E, notes, ped, start, mel_ids, S=None, offset=0.0):
    """Check every note that was sent: did that key sound (fine pitch analysis, wide window),
    exactly when (5 ms harmonic flux), and how loud. `offset` = measured constant latency."""
    L0 = 10 * np.log10(E + 1e-10); nf = L0.shape[1]            # each key's OWN pitch (no overtones)
    res = []
    starts = np.array([x["s"] for x in notes])
    for idx, n in enumerate(notes):
        k = n["p"] - 21
        if not 0 <= k < 88: continue
        t = n["s"] - start + offset
        i0 = int(round(t * SR / HOP))
        if i0 < 25 or i0 + 30 >= nf: continue
        # other notes struck or still held around this one (their harmonics could be mistaken for it)
        lo_i, hi_i = np.searchsorted(starts, n["s"] - 3.0), np.searchsorted(starts, n["s"] + 0.15)
        near = [x["p"] for x in notes[lo_i:hi_i] if x is not n and x["s"] >= n["s"] - 0.15 or
                (x is not n and x["s"] < n["s"] and sounding_until(x, ped) > n["s"] - 0.05)]
        # is this key's own pitch an overtone of something else sounding? then presence is ambiguous
        shadowed = any(n["p"] - q in (12, 19, 24, 28, 31, 36) for q in near)
        # look only until this same key is struck again (so the next strike can't count for this one)
        nxt = [x["s"] for x in notes[idx + 1: idx + 40] if x["p"] == n["p"] and x["s"] > n["s"] + 0.02]
        span = int(min(0.25, (min(nxt) - n["s"]) if nxt else 0.25) * SR / HOP)
        span = max(4, span)
        before = float(np.median(L0[k, i0 - 20: i0 - 3]))         # 200..30 ms before
        peak = float(L0[k, i0: i0 + span].max())
        floor_db = float(np.percentile(L0[k], 20))
        was_ringing = before > floor_db + 12                      # key already sounding (pedal / re-strike)
        rise = peak - before
        if rise >= 4.0 and not (shadowed and rise < 8.0): heard = True
        elif was_ringing and rise <= -1.0: heard = False          # it was ringing and just faded: no new strike
        elif was_ringing or shadowed: heard = None                # can't tell
        else: heard = False
        bass = n["p"] < 48
        edge = t < 1.0 or t > nf * HOP / SR - 1.5                 # too close to the start/end of the recording
        err_ms = None
        FM = note_flux(S, n["p"], [q for q in near if q != n["p"]]) if S is not None else None
        j0 = int(round(t * SR / T_HOP)); lo, hi = j0 - 16, j0 + 24              # -80 .. +120 ms
        if FM is not None and lo > 30 and hi + 60 < FM[0].shape[1]:
            F, M = FM
            # presence from this note's UNIQUE harmonics (no other note can fake these)
            span5 = max(4, span * HOP // T_HOP)
            lv_b = 20 * np.log10(np.median(M[:, j0 - 40: j0 - 6], axis=1) + 1e-9)
            lv_a = 20 * np.log10(M[:, j0: j0 + span5].max(1) + 1e-9)
            urise = float(np.median(lv_a - lv_b))
            if urise >= 6.0: heard = True
            elif urise <= 1.0 and heard is not True: heard = False
            elif urise <= 1.0 and rise < 8.0: heard = False                        # CQT rise came from a neighbour
            if heard:
                # time each unique harmonic separately; only trust it if they agree (within 15 ms)
                bg = np.median(F[:, max(0, j0 - 200): j0 + 200], axis=1) + 1e-9
                need = 3.0 if n["p"] >= 48 else 5.0
                ests = []
                for b in range(F.shape[0]):
                    e, h = leading_edge(F[b, lo:hi])
                    if h > need * bg[b]: ests.append(e)
                if len(ests) >= 2:
                    ests = np.array(ests); med_e = float(np.median(ests))
                    close = ests[np.abs(ests - med_e) <= 3]
                    if len(close) >= max(2, 0.6 * len(ests)):
                        err_ms = (float(np.mean(close)) - 16) * T_HOP / SR * 1000
        if heard is False and (bass or edge): heard = None       # bass pitches are too blurred in time to judge
        res.append(dict(i=idx, s=n["s"], p=n["p"], v=n["v"], melody=idx in mel_ids, heard=heard,
                        err_ms=None if err_ms is None else round(err_ms, 1), level_db=round(peak, 1), rise_db=round(rise, 1)))
    # melody vs backing, note by note: is each tune note louder than the backing keys struck with it?
    by_time = sorted(res, key=lambda r: r["s"])
    for r in by_time:
        if not r["melody"]: continue
        rivals = [x for x in by_time if not x["melody"] and abs(x["s"] - r["s"]) <= 0.06]
        if rivals:
            top = max(rivals, key=lambda x: x["level_db"])
            r["over_backing_db"] = round(r["level_db"] - top["level_db"], 1); r["rival"] = key_name(top["p"])
    return res


def section_timing(rec, notes, start, win=2.0, search_ms=120):
    """How early/late each 2-second stretch runs compared with what was sent: line up that stretch's
    strikes with the sent notes (cross-correlation over many notes -> precise to a few ms).
    Also: do soft notes fire later than loud ones (player-piano solenoids often do)?"""
    import librosa
    env = librosa.onset.onset_strength(y=rec, sr=SR, hop_length=T_HOP, lag=1, max_size=3)
    env = (env - np.median(env)) / (np.std(env) + 1e-9)
    nf = len(env); out = []
    k = np.exp(-np.arange(6) / 2.0)
    span = int(search_ms / 1000 * SR / T_HOP)
    end = start + nf * T_HOP / SR
    # only whole stretches with room to slide (edges of the recording would bias the result)
    for w0 in np.arange(start + search_ms / 1000 + 0.2, end - win - search_ms / 1000 - 0.2, win):
        wn = [n for n in notes if w0 <= n["s"] < w0 + win]
        if len(wn) < 6: continue
        a = int((w0 - start) * SR / T_HOP); b = a + int(win * SR / T_HOP)
        train = np.zeros(b - a + 2 * span)
        for n in wn:
            i = int(round((n["s"] - w0) * SR / T_HOP)) + span
            if 0 <= i < len(train): train[i] += (n["v"] / 127) ** 1.5
        train = np.convolve(train, k)[: len(train)]
        scores = {}
        for d in range(-span, span + 1):                          # try shifting the sent notes by d frames
            s0 = a + d - span
            if s0 < 0 or s0 + len(train) > nf: continue
            scores[d] = float(np.dot(env[s0: s0 + len(train)], train))
        if not scores: continue
        best = max(scores, key=scores.get)
        if abs(best) >= span - 1: continue                       # stuck at the search limit: not trustworthy
        frac = 0.0                                                # parabola through the best shift
        if best - 1 in scores and best + 1 in scores:
            y0, y1, y2 = scores[best - 1], scores[best], scores[best + 1]
            frac = float(np.clip(0.5 * (y0 - y2) / (y0 - 2 * y1 + y2 + 1e-9), -0.5, 0.5))
        out.append(dict(t=round(float(w0), 2), offset_ms=round((best + frac) * T_HOP / SR * 1000, 1), notes=len(wn)))
    if out:
        med = np.median([o["offset_ms"] for o in out])
        for o in out: o["drift_ms"] = round(o["offset_ms"] - med, 1)      # relative to the constant latency
    return out


def against_original(E, ref, start, dur):
    """Pitch by pitch against the original song: which notes does the song have that the piano doesn't
    (missing), and which does the piano play that the song doesn't (wrong/extra)? 100 ms grid, octave-folded."""
    if ref is None or "cqt" not in ref: return [], None
    hop = 0.1
    nf = int(dur / hop)
    # piano recording -> 12 pitch classes per 100 ms
    step = int(hop * SR / HOP)
    R = np.stack([E[:, i * step:(i + 1) * step].mean(1) for i in range(nf)], 1)     # [88, nf]
    rc = np.zeros((12, nf))
    for k in range(88): rc[(k + 21) % 12] += R[k]
    # original (stored as dB on its own grid) -> same grid
    G = 10 ** (ref["cqt"].astype(np.float32) / 10)                                   # [88, frames]
    rh = ref["cqt_hop"]; oc = np.zeros((12, nf))
    for i in range(nf):
        a = int((start + i * hop) / rh); b = max(a + 1, int((start + (i + 1) * hop) / rh))
        if b > G.shape[1]: break
        g = G[:, a:b].mean(1)
        for k in range(88): oc[(k + 21) % 12, i] += g[k]
    import scipy.ndimage as ndi
    norm = lambda X: ndi.median_filter(X / (X.max(0, keepdims=True) + 1e-12), size=(1, 3))
    rn, on = norm(rc), norm(oc)
    loud_o = oc.sum(0) > 0.02 * np.percentile(oc.sum(0), 95)
    loud_r = rc.sum(0) > 0.02 * np.percentile(rc.sum(0), 95)
    missing = (on >= 0.5) & (rn < 0.12) & loud_o & loud_r
    extra = (rn >= 0.5) & (on < 0.12) & loud_o & loud_r
    out = []
    for kind, M in (("missing", missing), ("extra", extra)):
        for pc in range(12):
            row = M[pc]; i = 0
            while i < nf:
                if row[i]:
                    j = i
                    while j < nf and row[j]: j += 1
                    if (j - i) * hop >= 0.3:                                      # at least 300 ms
                        out.append(dict(kind=kind, pc=NOTE_NAMES[pc], t0=round(start + i * hop, 2),
                                        t1=round(start + j * hop, 2)))
                    i = j
                else:
                    i += 1
    agree = float((np.minimum(rn, on).sum(0) / (np.maximum(rn, on).sum(0) + 1e-9))[loud_o & loud_r].mean()) \
        if (loud_o & loud_r).any() else None
    return sorted(out, key=lambda x: x["t0"]), agree


def viz_payload(E, ref, notes, ped, start, dur, hop=0.04):
    """Three aligned pictures (88 keys x time): how it should sound (original), what the piano played
    (recording), and the notes that were sent - as 0..255 images for the browser."""
    import base64
    nf = int(dur / hop); step = hop * SR / HOP
    def to_u8(dbs):
        lo, hi = np.percentile(dbs, 50), np.percentile(dbs, 99.7)
        return (np.clip((dbs - lo) / (hi - lo + 1e-9), 0, 1) * 255).astype(np.uint8)
    H = np.stack([E[:, int(i * step): int((i + 1) * step) + 1].mean(1) for i in range(nf)], 1)
    heard = to_u8(10 * np.log10(H + 1e-10))
    sent = np.zeros((88, nf), np.uint8)
    for n in notes:
        a = int((n["s"] - start) / hop); b = int((sounding_until(n, ped) - start) / hop) + 1
        k = n["p"] - 21
        if 0 <= k < 88 and b > 0 and a < nf: sent[k, max(0, a):min(nf, b)] = max(60, n["v"] * 2)
    exp = None
    if ref is not None and "cqt" in ref:
        rh = ref["cqt_hop"]; G = ref["cqt"].astype(np.float32)
        cols = [G[:, min(G.shape[1] - 1, int((start + i * hop) / rh))] for i in range(nf)]
        exp = to_u8(np.stack(cols, 1))
    b64 = lambda X: base64.b64encode(np.ascontiguousarray(X.T).tobytes()).decode()   # row = one time step
    return dict(hop=hop, frames=nf, keys=88, start=round(start, 3), heard=b64(heard), sent=b64(sent),
                expected=b64(exp) if exp is not None else None)


# ------------------------------------------------------------------------------ the check
def analyse(rec, midi, name="song", hint=None, ref_path=None, vol=None, floor=None):
    import librosa
    notes, ped, song_len = read_midi(midi)
    rec = rec.astype(np.float32); rec = rec / (np.abs(rec).max() + 1e-9)
    dur = len(rec) / SR
    start, conf = robust_align(rec, notes, song_len, hint)
    if start < 0:                                       # recording began before the song: drop that lead-in
        rec = rec[int(round(-start * SR)):]; start = 0.0; dur = len(rec) / SR
    rec_on = librosa.onset.onset_detect(y=rec, sr=SR, hop_length=HOP, units="time", backtrack=False, delta=0.05) + start
    seg = [n for n in notes if start - 0.5 <= n["s"] < start + dur]
    mel = melody_line(seg, ped)
    E = note_energy(rec); nf = E.shape[1]
    C_rec = chroma(rec); C_sent = roll_chroma(seg, ped, start, C_rec.shape[1])
    ref = load_ref(ref_path)
    windows = []
    for w0 in np.arange(start, start + dur - 1.0, WINDOW):
        w1 = min(w0 + WINDOW, start + dur)
        wn = [n for n in seg if w0 <= n["s"] < w1]
        if not wn:
            windows.append(dict(t=round(float(w0), 1), silent=True)); continue
        # timing: each sent note start -> nearest heard note start
        st = np.array([n["s"] for n in wn]); ro = rec_on[(rec_on > w0 - 0.2) & (rec_on < w1 + 0.2)]
        if len(ro):
            d = ro[np.clip(np.searchsorted(ro, st), 0, len(ro) - 1)] - st
            d2 = ro[np.clip(np.searchsorted(ro, st) - 1, 0, len(ro) - 1)] - st
            d = np.where(np.abs(d) < np.abs(d2), d, d2)
            heard = np.abs(d) <= 0.07
            late_ms = float(np.median(d[np.abs(d) <= 0.15]) * 1000) if (np.abs(d) <= 0.15).any() else None
            heard_share = float(heard.mean())
        else:
            late_ms, heard_share = None, 0.0
        # notes: what you hear vs what was sent
        a, b = int((w0 - start) * SR / 1024), int((w1 - start) * SR / 1024)
        sent_vs_heard = float((C_rec[:, a:b] * C_sent[:, a:b]).sum(0).mean()) if b > a else None
        # melody: energy at the tune's keys vs the other keys sounding at the same moments
        ratios = []
        for m in [m for m in mel if w0 <= m["s"] < w1]:
            i0, i1 = int((m["s"] - start) * SR / HOP) + 2, int((m["s"] - start) * SR / HOP) + 12
            if i1 >= nf or i0 < 0: continue
            mk = m["p"] - 21
            if not 0 <= mk < 88: continue
            around = [n for n in seg if n is not m and n["s"] <= m["s"] + 0.05 and sounding_until(n, ped) > m["s"]
                      and n["p"] != m["p"] and 0 <= n["p"] - 21 < 88 and abs(n["p"] - m["p"]) != 12]
            if not around: continue
            # only judge notes that were SENT as the tune (clearly louder than what's around them);
            # otherwise it's just the top note of a chord or lick, not a melody
            if m["v"] < max(n["v"] for n in around) + 10: continue
            others = [n["p"] - 21 for n in around]
            em = E[mk, i0:i1].mean(); eo = np.mean([E[k, i0:i1].mean() for k in others])
            ratios.append(10 * np.log10((em + 1e-12) / (eo + 1e-12)))
        melody_db = float(np.median(ratios)) if ratios else None
        like_song = None
        if ref is not None:
            ra, rb = int(w0 / ref["hop"]), int(w1 / ref["hop"])
            rc = ref["chroma"][:, ra:rb]; k = min(rc.shape[1], b - a)
            if k > 2: like_song = float((C_rec[:, a:a + k] * rc[:, :k]).sum(0).mean())
        windows.append(dict(t=round(float(w0), 1), notes_sent=len(wn), heard_on_time=round(heard_share, 2),
                            timing_ms=None if late_ms is None else round(late_ms),
                            sounds_like_sent=None if sent_vs_heard is None else round(sent_vs_heard, 2),
                            melody_over_backing_db=None if melody_db is None else round(melody_db, 1),
                            sounds_like_song=None if like_song is None else round(like_song, 2),
                            notes_per_sec=round(len(wn) / (w1 - w0), 1)))
    # ------------------------------------------------ problems, in plain words
    problems = []
    for w in windows:
        if w.get("silent"): continue
        why = []
        if w["melody_over_backing_db"] is not None and w["melody_over_backing_db"] < 3:
            why.append(f"melody buried (only {w['melody_over_backing_db']:+.0f} dB over the backing)")
        if w["timing_ms"] is not None and abs(w["timing_ms"]) > 40:
            why.append(f"notes {'late' if w['timing_ms'] > 0 else 'early'} by ~{abs(w['timing_ms'])} ms")
        if w["heard_on_time"] < 0.5:
            why.append(f"only {w['heard_on_time']:.0%} of notes heard on time (missed or smeared)")
        if w["sounds_like_sent"] is not None and w["sounds_like_sent"] < 0.6:
            why.append("what the piano plays doesn't match the notes sent (dropped/muddy notes)")
        if w["sounds_like_song"] is not None and w["sounds_like_song"] < 0.55:
            why.append("doesn't sound like the original song here (wrong chords/notes in the arrangement)")
        if w["notes_per_sec"] > 14:
            why.append(f"very busy ({w['notes_per_sec']} notes/s) - may sound jumbled")
        if why: problems.append(dict(time=fmt(w["t"]), t=w["t"], issues=why))
    # ------------------------------------------------ note-by-note (precise)
    mel_ids = set()
    for m in mel:
        around = [x for x in seg if x is not m and abs(x["s"] - m["s"]) <= 0.06]
        if not around or m["v"] >= max(x["v"] for x in around) + 10:
            mel_ids.add(next(i for i, x in enumerate(seg) if x is m))
    S = stft_mag(rec)
    first = note_by_note(E, seg, ped, start, mel_ids, S)
    errs0 = [r["err_ms"] for r in first if r["err_ms"] is not None]
    latency_ms = float(np.median(errs0)) if errs0 else 0.0     # constant delay (MIDI -> hammer -> mic): not an error
    nb = note_by_note(E, seg, ped, start, mel_ids, S, offset=latency_ms / 1000)
    vs_orig, orig_agree = against_original(E, ref, start, dur)
    sections = section_timing(rec, seg, start)
    issues = []
    for s_ in sections:
        if abs(s_["drift_ms"]) > 20:
            issues.append(dict(t=s_["t"], kind="timing (stretch)", text=f"this 2 s stretch runs {abs(s_['drift_ms']):.0f} ms "
                               f"{'late' if s_['drift_ms'] > 0 else 'early'} compared with the rest ({s_['notes']} notes)"))
    # do soft notes fire later than loud ones? (median timing of confidently-timed notes by velocity)
    timed_v = [(r["v"], r["err_ms"]) for r in nb if r["err_ms"] is not None]
    vel_lag = None
    if len(timed_v) >= 20:
        tv = np.array(timed_v); soft, loud = tv[tv[:, 0] < np.median(tv[:, 0])], tv[tv[:, 0] >= np.median(tv[:, 0])]
        if len(soft) >= 8 and len(loud) >= 8:
            vel_lag = round(float(np.median(soft[:, 1]) - np.median(loud[:, 1])), 1)
    for r in nb:
        nm, t = key_name(r["p"]), r["s"]
        if r["heard"] is False:
            issues.append(dict(t=round(t, 2), kind="not heard", text=f"{'melody ' if r['melody'] else ''}{nm} didn't sound (sent at velocity {r['v']})"))
        # (single-note timing stays in the data/visual only: in dense chords it isn't reliable enough to list)
        if r.get("over_backing_db") is not None and r["over_backing_db"] < 0:
            issues.append(dict(t=round(t, 2), kind="melody buried",
                               text=f"melody {nm} is {-r['over_backing_db']:.0f} dB quieter than the backing {r['rival']}"))
    for x in vs_orig:
        if x["kind"] == "missing":
            issues.append(dict(t=x["t0"], kind="missing vs song", text=f"the song has {x['pc']} here ({x['t1'] - x['t0']:.1f}s) but the piano doesn't"))
        else:
            culprits = sorted({key_name(n["p"]) for n in seg if NOTE_NAMES[n["p"] % 12] == x["pc"]
                               and n["s"] < x["t1"] and sounding_until(n, ped) > x["t0"]})
            issues.append(dict(t=x["t0"], kind="wrong vs song", text=f"piano plays {x['pc']} ({x['t1'] - x['t0']:.1f}s) that the song doesn't have here"
                               + (f" - sent {', '.join(culprits)}" if culprits else " (not sent: ringing/sympathetic)")))
    issues.sort(key=lambda x: x["t"])
    for x in issues: x["time"] = f"{int(x['t'] // 60)}:{x['t'] % 60:05.2f}"
    heard_n = [r for r in nb if r["heard"]]
    judged = [r for r in nb if r["heard"] is not None]
    timed = [r for r in heard_n if r["err_ms"] is not None]
    errs = np.array([abs(r["err_ms"]) for r in timed]) if timed else np.array([np.nan])
    mel_r = [r for r in nb if r.get("over_backing_db") is not None]
    precise = dict(notes_checked=len(nb), notes_heard=round(len(heard_n) / max(1, len(judged)), 3),
                   notes_unclear=sum(1 for r in nb if r["heard"] is None), latency_ms=round(latency_ms, 1),
                   soft_notes_lag_ms=vel_lag,
                   stretch_timing_spread_ms=round(float(np.std([s_["drift_ms"] for s_ in sections])), 1) if sections else None,
                   timing_median_ms=round(float(np.nanmedian(errs)), 1), timing_p90_ms=round(float(np.nanpercentile(errs, 90)), 1),
                   melody_notes_on_top=round(sum(r["over_backing_db"] >= 0 for r in mel_r) / max(1, len(mel_r)), 3) if mel_r else None,
                   pitch_agreement_with_song=None if orig_agree is None else round(orig_agree, 3))
    viz = viz_payload(E, ref, seg, ped, start, dur)

    good = [w for w in windows if not w.get("silent")]
    summary = dict(
        song=name, recorded_seconds=round(dur, 1), starts_at=fmt(start), alignment_confidence=round(conf, 1),
        alignment_ok=conf > 6,
        melody_over_backing_db=med([w["melody_over_backing_db"] for w in good]),
        timing_ms=med([w["timing_ms"] for w in good]), heard_on_time=med([w["heard_on_time"] for w in good]),
        sounds_like_sent=med([w["sounds_like_sent"] for w in good]), sounds_like_song=med([w["sounds_like_song"] for w in good]),
        jukebox_volume=vol, jukebox_softest=floor, problem_sections=len(problems), checked_sections=len(good), **precise)
    return dict(summary=summary, problems=problems, windows=windows, issues=issues, notes=nb, viz=viz, sections=sections)


def load_ref(path):
    if not path or not Path(path).exists(): return None
    d = np.load(path)
    ref = dict(chroma=d["chroma"], hop=float(d["hop"]))
    if "cqt" in d: ref.update(cqt=d["cqt"], cqt_hop=float(d["cqt_hop"]))
    return ref


def cqt88_db(y, hop_s=0.02):
    """The original's energy at each of the 88 keys (dB, float16) for the 'should sound like' picture."""
    import librosa
    hop = int(hop_s * SR)
    C = np.abs(librosa.cqt(y, sr=SR, hop_length=hop, fmin=librosa.midi_to_hz(21) / 2 ** (1 / 36),
                           n_bins=88 * 3, bins_per_octave=36)) ** 2
    return (10 * np.log10(C.reshape(88, 3, -1).max(1) + 1e-10)).astype(np.float16)


def med(xs):
    xs = [x for x in xs if x is not None]
    return round(float(np.median(xs)), 2) if xs else None


def fmt(t):
    return f"{int(t // 60)}:{int(t % 60):02d}"


def base_name(name):
    return re.sub(r"\s*\((auto|pop|piano|tuned|arranged[^)]*|\d+)\)\s*$", "", Path(name).stem).strip()


def ref_for(name):
    p = REFS / f"{base_name(name)}.npz"
    return p if p.exists() else None


def text_report(r):
    s = r["summary"]; L = []
    L.append(f"Piano check: {s['song']}  ({s['recorded_seconds']}s recorded, starting at {s['starts_at']} in the song)")
    if not s["alignment_ok"]:
        L.append("  ! Couldn't confidently line the recording up with the song - was the piano playing this song, loud enough?")
    def line(label, v, unit=""):
        if v is not None: L.append(f"  {label:34s} {v}{unit}")
    line("Melody louder than backing (median)", s["melody_over_backing_db"], " dB   (want 6+)")
    line("Timing (median, + = piano late)", s["timing_ms"], " ms   (want within ±30)")
    line("Notes heard on time", None if s["heard_on_time"] is None else f"{s['heard_on_time']:.0%}")
    line("Sounds like the notes sent", s["sounds_like_sent"], "   (1.0 = perfect)")
    line("Sounds like the original song", s["sounds_like_song"], "   (1.0 = perfect)")
    L.append(f"  {s['problem_sections']} of {s['checked_sections']} sections have problems:")
    for p in r["problems"]:
        L.append(f"    {p['time']}  " + "; ".join(p["issues"]))
    if s.get("notes_checked"):
        L.append(f"  Note by note: {s['notes_checked']} notes checked, {s['notes_heard']:.0%} sounded, timing median "
                 f"{s['timing_median_ms']} ms (90% within {s['timing_p90_ms']} ms)"
                 + (f", melody on top for {s['melody_notes_on_top']:.0%} of tune notes" if s.get("melody_notes_on_top") is not None else "")
                 + (f", pitch agreement with the song {s['pitch_agreement_with_song']:.0%}" if s.get("pitch_agreement_with_song") is not None else ""))
        L.append(f"  {len(r.get('issues', []))} precise issues:")
        for x in r.get("issues", []):
            L.append(f"    {x['time']}  [{x['kind']}] {x['text']}")
    return "\n".join(L)


def save(r, rec=None):
    CHECKS.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    base = CHECKS / f"{base_name(r['summary']['song'])} {stamp}"
    (base.with_suffix(".json")).write_text(json.dumps(r, indent=1), encoding="utf-8")
    (base.with_suffix(".txt")).write_text(text_report(r), encoding="utf-8")
    if rec is not None:
        import soundfile as sf
        sf.write(str(base) + ".wav", rec, SR)
    return base


def save_reference(original_audio_16k, name):
    """Called by the converter: remember what the original song sounds like, for later checks."""
    REFS.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(REFS / f"{base_name(name)}.npz", chroma=chroma(original_audio_16k).astype(np.float32), hop=1024 / SR,
                        cqt=cqt88_db(original_audio_16k), cqt_hop=0.02)


def reference_for_browser(name, hop=0.04):
    """The whole song's 'should sound like' picture for the live view (88 keys x time, 0..255)."""
    import base64
    ref = load_ref(ref_for(name))
    if ref is None or "cqt" not in ref: return None
    G = ref["cqt"].astype(np.float32); step = max(1, int(round(hop / ref["cqt_hop"])))
    n = G.shape[1] // step
    G = G[:, : n * step].reshape(88, n, step).mean(2)
    lo, hi = np.percentile(G, 50), np.percentile(G, 99.7)
    u8 = (np.clip((G - lo) / (hi - lo + 1e-9), 0, 1) * 255).astype(np.uint8)
    return dict(hop=round(ref["cqt_hop"] * step, 4), frames=n, keys=88, data=base64.b64encode(np.ascontiguousarray(u8.T).tobytes()).decode())


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--midi", required=True)
    ap.add_argument("--file", help="analyse this recording instead of recording now")
    ap.add_argument("--seconds", type=float, default=60)
    ap.add_argument("--at", type=float, help="roughly where in the song the recording starts (seconds)")
    a = ap.parse_args()
    if a.file:
        import librosa
        rec, _ = librosa.load(a.file, sr=SR, mono=True)
    else:
        import sounddevice as sd
        print(f"Recording {a.seconds:.0f}s from {sd.query_devices(kind='input')['name']} - play the song on the piano now...")
        rate = int(sd.query_devices(kind="input")["default_samplerate"])
        x = sd.rec(int(a.seconds * rate), samplerate=rate, channels=1, dtype="float32"); sd.wait()
        import librosa
        rec = librosa.resample(x[:, 0], orig_sr=rate, target_sr=SR)
    r = analyse(rec, Path(a.midi), name=Path(a.midi).name, hint=a.at, ref_path=ref_for(a.midi))
    base = save(r, rec)
    print(text_report(r)); print(f"\nSaved: {base}.txt / .json / .wav")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile piano_profile.json
{
  "_about": "Calibration for the player piano this jukebox drives. Measured with the piano check (song-to-midi/piano_check.py) over 12 real recordings of 'Dancing in the Moonlight' (arranged v6..v18), 19 Nov 2026. See docs/CALIBRATION.md for how each number was found. Edit these if you change the jukebox settings or use a different piano.",

  "jukebox_volume": 0.6,
  "_jukebox_volume": "Jukebox Settings -> Volume. The jukebox plays velocity * volume. 0.35 left the melody only 9 velocity steps above the backing; 0.6 gives room (melody ~74 vs backing ~50).",
  "jukebox_softest": 35,
  "_jukebox_softest": "Jukebox Settings -> Softest note. Anything below this is raised to it.",

  "levels": {
    "_about": "Target loudness ON THE PIANO (after volume), not MIDI velocity. The arranger converts.",
    "melody_verse": 73,
    "melody_chorus": 75,
    "melody_octave_double": 60,
    "melody_light_break": 60,
    "bass": 56,
    "bass_chorus": 60,
    "chords": 52,
    "chords_chorus": 58,
    "keys_part": 58,
    "light_figure": 52,
    "under_melody_cap": 52,
    "minimum": 45,
    "_minimum": "Notes below ~36 on the piano failed 22% of the time; 45+ is reliable."
  },

  "weak_keys": [48, 51, 53, 58, 60, 62, 63, 65],
  "_weak_keys": "C3 Eb3 F3 Bb3 C4 D4 Eb4 F4: repeatedly didn't sound at 41-52. Eb4 (63) was the worst.",
  "weak_key_minimum": 58,
  "weak_key_overrides": {"63": 62, "62": 58},

  "min_note_seconds": 0.15,
  "_min_note_seconds": "Notes shorter than 0.06 s failed 10.7% of the time (vs ~2% for >= 0.15 s): the key lifts before the hammer lands.",
  "key_reset_seconds": 0.12,
  "_key_reset_seconds": "A key must be released this long before it is struck again. At 80 ms most failed repeats came exactly 79-80 ms after release.",
  "merge_same_key_within": 0.2,
  "melody_repeat_within": 0.15,

  "soft_note_lag": [[45, 0.027], [60, 0.020]],
  "_soft_note_lag": "Solenoids fire soft notes late. [played_velocity_up_to, seconds_to_send_early]. Measured: soft notes 13-14 ms late after a 13 ms correction (~27 ms total at <= 45); verified to land within 2 ms.",

  "max_keys_ringing": 12,
  "_max_keys_ringing": "Above ~13 keys ringing at once (pedal included) it sounds jumbled.",

  "pedal": {
    "change_at": ["melody_on", "melody_off", "bass_on"],
    "min_press": 0.12,
    "max_span": 4.0,
    "_about": "Holding the pedal through chords (96% of the song) made every melody note ring ~2 s and overlap -> 'jumbled'. No pedal sounded dry. Changing at every tune note, tune note end and bass note keeps the tune clean and the backing warm."
  },

  "section_timing_correction": false,
  "_section_timing_correction": "Do NOT pre-shift late-sounding sections: what looked like late stretches was the soft-note lag, and correcting both over-shot by 20-70 ms."
}

In [ ]:
#@title 2. Settings
mode = "auto"  #@param ["auto", "piano", "pop"]
#@markdown **auto** - arranges the song for your player piano (recommended; uses piano_profile.json) ·
#@markdown **piano** - for songs that are only piano · **pop** - drums removed, you pick the parts below
keep_melody = True  #@param {type:"boolean"}
keep_chords = True  #@param {type:"boolean"}
keep_bass = False  #@param {type:"boolean"}
parts = [p for p, on in (("vocals", keep_melody), ("other", keep_chords), ("bass", keep_bass)) if on] or ["vocals", "other"]
print("Mode:", mode, "| parts (pop mode):", parts)

In [ ]:
#@title 3. Upload songs and convert
import importlib, time, warnings
warnings.filterwarnings("ignore")
from google.colab import files
import song2midi; importlib.reload(song2midi)
song2midi.OUT = song2midi.Path("/content/output")

uploaded = files.upload()
for fname in uploaded:
    t0 = time.time()
    print(f"\n=== {fname} ===")
    audio, rate = song2midi.load_file(fname)
    midi_path = song2midi.output_path(song2midi.Path(fname).stem, mode)
    summary = song2midi.convert(audio, rate, mode, parts, midi_path, progress=False)
    print(f"Finished in {time.time() - t0:.0f}s: {summary}")
    files.download(str(midi_path))
    os.remove(fname)